In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

@dp.table(name="Bronze_Table_Transaction")
def bronze_table():
    df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/etl/transaction/raw_transaction/")
    
    df.write.mode("overwrite").saveAsTable("etl.transaction.combined_transaction")

    bronze_df= spark.read.table("etl.transaction.combined_transaction")
    return bronze_df

@dp.table(name="Silver_Table_Transaction")
def silver_table():
    bronze_df = spark.read.table("Bronze_Table_Transaction")

    silver_df=bronze_df.withColumn("transaction_date",to_date(col("transaction_date"),"MM-dd-yyyy"))
    #display(silver_df)

    #trim white space
    silver_df = silver_df.withColumn(
        "product_name",
        initcap (
            trim(
                (regexp_replace
                (col("product_name"), r"\s+", " ")
                )
                )
    )
    )

    silver_df = silver_df.withColumn(
        "product_name",
        regexp_replace(col("product_name"), r"(?i)\busb-c\b", "USB-C")
    )

    silver_df = silver_df.withColumn(
        "product_name",
        regexp_replace(col("product_name"), r"(?i)\bt-shirt\b", "T-Shirt")
    )

    return silver_df


@dp.table(name="Gold_Products_Sold")
def gold_table():
    silver_df = spark.read.table("Silver_Table_Transaction")
    return (
    silver_df
    .groupBy("product_name")
    .agg(
        sum("quantity").alias("units_sold"),
        round(sum("total_amount"),2).alias("total_sales")
    )
    .orderBy("total_sales", ascending=False)
)

@dp.table(name="Gold_Products_Sold_by_Category")
def gold_products_sold_by_category():
    silver_df = spark.read.table("Silver_Table_Transaction")
    return (
    silver_df
    .groupBy("category")
    .agg(
        round(sum("total_amount"),2).alias("total_sales"),
        sum("quantity").alias("units_sold")
    )
    .orderBy("total_sales", ascending=False)
)

@dp.table(name="Gold_Top_Store")
def gold_top_store():
    silver_df = spark.read.table("Silver_Table_Transaction")
    
    return(
        silver_df
        .groupBy("store_location")
        .agg(
            round(sum("total_amount"),2).alias("total_sales")
            )
        .orderBy("total_sales", ascending=False)
    )

